# Clinical Document Classification — Traditional Machine Learning

Bachelor thesis: *Clinical Document Classification Using Machine Learning* (Myriam Joseph, 52-6738).

This notebook covers the **traditional ML** branch of the comparative analysis described in the thesis
(sections 4.1-4.3.1, 5.2). Five categories of clinical document are classified: `Discharge Summary`,
`Gastroenterology`, `General Medicine`, `Neurology`, `Radiology`.

Pipeline: **load data -> EDA -> text preprocessing -> TF-IDF feature engineering -> train/compare
traditional classifiers -> export the best one (Voting Classifier) for the web app.**

The preprocessing steps mirror `src/preprocessing.py`, which is shared with the sequence-model and
BERT notebooks and with the deployed backend (`app/backend/myapp/clinical_document_classifier.py`), so
a document is cleaned identically everywhere in the project.

Related deliverable: the exported `voting_classifier_model.joblib` + `tfidf_vectorizer.joblib` from this
notebook are what `CliniDoc Predictor` (see `../app`) actually serves.


In [ ]:
import sys
sys.path.append('src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectFromModel, SelectPercentile, chi2
from sklearn.svm import SVC, LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn import metrics

from preprocessing import (
    CLASS_NAMES,
    load_clinical_documents,
    load_spacy_model,
    preprocess_document,
)

RANDOM_STATE = 133
DATA_DIR = "../data/clinical_documents"  # one sub-folder per class, see project README

## 1. Load the dataset

In [ ]:
df = load_clinical_documents(DATA_DIR)
print(df.shape)
df.head()

## 2. Exploratory Data Analysis

Class balance and a quick look at the raw text before any cleaning.

In [ ]:
df["label"].value_counts().plot(kind="bar", title="Documents per category")
plt.ylabel("count")
plt.show()

In [ ]:
df.isnull().sum()

In [ ]:
for i in range(3):
    print(f"[{df['label'][i]}]")
    print(df["document"][i][:400], "...\n")

## 3. Text Preprocessing & Cleaning

Four steps, applied in order (see `src/preprocessing.py` for the full implementation):

1. **Abbreviation expansion** - clinical shorthand (`MRI`, `HTN`, `CBC`, ...) spelled out.
2. **De-identification** - patient/doctor/hospital names and titles stripped.
3. **Regex cleanup + lemmatization** - dates, digits, punctuation removed; spaCy lemmatizer applied.
4. **Stopword removal** - a general English stopword list plus a domain-specific medical one
   (words like *patient*, *hospital*, *document* appear in almost every class and add no signal).

In [ ]:
nlp = load_spacy_model("en_core_web_lg")
print("stopwords in vocabulary:", len(nlp.Defaults.stop_words))

In [ ]:
before = df["document"][0]

df["clean_document"] = [
    preprocess_document(doc, nlp) for doc in df["document"]
]

print("BEFORE:\n", before[:400])
print("\nAFTER:\n", df["clean_document"][0][:400])

## 4. Feature Engineering — TF-IDF

Term Frequency - Inverse Document Frequency turns each cleaned document into a sparse numeric
vector, weighting words that are frequent in one document but rare across the corpus more heavily. This is the representation the deployed model uses.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    df["clean_document"], df["label"], test_size=0.3, random_state=RANDOM_STATE
)

vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(x_train)
X_test = vectorizer.transform(x_test)

print("Train:", X_train.shape, " Test:", X_test.shape)

## 5. Feature selection (exploratory)

Three feature-selection strategies were tried to see whether trimming the TF-IDF vocabulary down to
the most discriminative terms would help: importance-based selection from a Random
Forest, a chi-squared percentile filter, and an L1/L2-penalized linear SVM selector.

None of these improved on using the full TF-IDF vocabulary for the models below, so the final
pipeline (and the deployed model) uses the **full** `X_train`/`X_test` from step 4. This section is
kept to document what was evaluated, not because its output feeds into the final models.

In [ ]:
selector_rf = SelectFromModel(RandomForestClassifier(n_estimators=262, random_state=33))
X_train_rf_sel = selector_rf.fit_transform(X_train, y_train)
print("Random-Forest-importance selection:", X_train_rf_sel.shape, "features (from", X_train.shape[1], ")")

selector_chi2 = SelectPercentile(score_func=chi2, percentile=70)
X_train_chi2_sel = selector_chi2.fit_transform(X_train, y_train)
print("Chi2 70th-percentile selection:    ", X_train_chi2_sel.shape, "features")

selector_svc = SelectFromModel(LinearSVC(penalty="l2", dual=False))
X_train_svc_sel = selector_svc.fit_transform(X_train, y_train)
print("Linear-SVC-based selection:        ", X_train_svc_sel.shape, "features")

## 6. Classification models

A comparative sweep across traditional ML algorithms, each tuned with
`GridSearchCV` (5-fold) where noted. A single helper collects predictions and prints/stores the
same metrics (confusion matrix, precision/recall/F1 macro-average, full classification report,
accuracy) for every model so results are directly comparable.

In [ ]:
results = []

def evaluate_and_report(name, model, X_te, y_te):
    predictions = model.predict(X_te)

    precision = metrics.precision_score(y_te, predictions, average="macro")
    recall = metrics.recall_score(y_te, predictions, average="macro")
    f1 = metrics.f1_score(y_te, predictions, average="macro")
    accuracy = metrics.accuracy_score(y_te, predictions)

    print(f"=== {name} ===")
    print("Accuracy: %.3f | Precision: %.3f | Recall: %.3f | F1: %.3f" % (accuracy, precision, recall, f1))
    print(metrics.classification_report(y_te, predictions))

    cm = metrics.confusion_matrix(y_te, predictions)
    sns.heatmap(cm, center=True, cmap="YlGnBu", annot=False)
    plt.title(f"{name} — confusion matrix")
    plt.show()

    results.append({"model": name, "accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1})
    return predictions

### 6.1 Support Vector Machine (SVM)

In [ ]:
svm_param_grid = {
    "C": [0.1, 1, 10, 100],
    "decision_function_shape": ["ovo", "ovr"],
    "class_weight": [None, "balanced"],
    "gamma": ["scale", "auto"],
}
svm_search = GridSearchCV(SVC(kernel="sigmoid", probability=True), svm_param_grid, cv=5)
svm_search.fit(X_train, y_train)
print("Best SVM params:", svm_search.best_params_)

In [ ]:
svm_clf = SVC(C=1, kernel="sigmoid", class_weight="balanced", decision_function_shape="ovo", probability=True)
svm_clf.fit(X_train, y_train)
_ = evaluate_and_report("SVM", svm_clf, X_test, y_test)

### 6.2 Naive Bayes

In [ ]:
nb_clf = MultinomialNB(alpha=0.1, fit_prior=False)
nb_clf.fit(X_train, y_train)
_ = evaluate_and_report("Naive Bayes", nb_clf, X_test, y_test)

### 6.3 Random Forest

In [ ]:
rf_param_grid = {
    "n_estimators": np.arange(50, 301, 10),
    "bootstrap": [True, False],
}
rf_search = GridSearchCV(RandomForestClassifier(random_state=RANDOM_STATE), rf_param_grid, cv=5)
rf_search.fit(X_train, y_train)
print("Best RF params:", rf_search.best_params_)

In [ ]:
rf_clf = RandomForestClassifier(n_estimators=110, n_jobs=6, random_state=RANDOM_STATE)
rf_clf.fit(X_train, y_train)
_ = evaluate_and_report("Random Forest", rf_clf, X_test, y_test)

### 6.4 Multi-Layer Perceptron (MLP)

In [ ]:
mlp_clf = MLPClassifier(random_state=RANDOM_STATE, max_iter=300)
mlp_clf.fit(X_train, y_train)
_ = evaluate_and_report("MLP", mlp_clf, X_test, y_test)

### 6.5 K-Nearest Neighbors (KNN)

In [ ]:
knn_param_grid = {
    "n_neighbors": np.arange(10, 71, 5),
    "weights": ["uniform", "distance"],
    "leaf_size": [20, 30, 40],
    "p": [1, 2],
    "metric": ["euclidean", "manhattan"],
}
knn_search = GridSearchCV(KNeighborsClassifier(), knn_param_grid, cv=5)
knn_search.fit(X_train, y_train)
print("Best KNN params:", knn_search.best_params_)

In [ ]:
knn_clf = KNeighborsClassifier(leaf_size=20, metric="euclidean", n_neighbors=10, p=1, weights="distance")
knn_clf.fit(X_train, y_train)
_ = evaluate_and_report("KNN", knn_clf, X_test, y_test)

### 6.6 Voting Classifier — SVM + KNN (best model)

Combining the tuned SVM and KNN into a hard-voting ensemble outperformed every
individual model and every sequence/transformer model tried (accuracy 0.89, F1 0.888 — see
`../docs/thesis.md` Table 5.16). Note the ensemble re-tunes `n_neighbors` for KNN as a component
(55 rather than 10) since a grid search over voting weight/strategy combinations found that
setting worked best *in combination*, even though 10 was best for KNN alone.

In [ ]:
voting_param_grid = {
    "voting": ["hard", "soft"],
    "weights": [[2, 2], [1, 1], [1, 2], [2, 1]],
}
voting_search = GridSearchCV(
    VotingClassifier(estimators=[
        ("knn", KNeighborsClassifier(leaf_size=20, metric="euclidean", n_neighbors=55, p=1)),
        ("svc", SVC(C=1, class_weight="balanced", decision_function_shape="ovo", kernel="sigmoid", probability=True)),
    ]),
    voting_param_grid,
    cv=5,
)
voting_search.fit(X_train, y_train)
print("Best Voting Classifier params:", voting_search.best_params_)

In [ ]:
voting_clf = VotingClassifier(
    estimators=[
        ("knn", KNeighborsClassifier(leaf_size=20, metric="euclidean", n_neighbors=55, p=1)),
        ("svc", SVC(C=1, class_weight="balanced", decision_function_shape="ovo", kernel="sigmoid", probability=True)),
    ],
    voting="hard",
)
voting_clf.fit(X_train, y_train)
_ = evaluate_and_report("Voting Classifier (SVM + KNN)", voting_clf, X_test, y_test)

## 7. Model comparison

Matches thesis Table 5.9 (per-model accuracy/precision/recall/F1).

In [ ]:
results_df = pd.DataFrame(results).set_index("model").sort_values("f1", ascending=False)
display(results_df)

results_df[["accuracy", "f1"]].plot(kind="bar", figsize=(8, 4), title="Traditional ML model comparison")
plt.ylabel("score")
plt.ylim(0, 1)
plt.show()

## 8. Export the best model

The Voting Classifier + its TF-IDF vectorizer are exported here and are the exact artifacts
shipped with the backend at `../app/backend/myapp/ml_artifacts/`.

In [ ]:
import os
os.makedirs("models", exist_ok=True)

joblib.dump(voting_clf, "models/voting_classifier_model.joblib")
joblib.dump(vectorizer, "models/tfidf_vectorizer.joblib")

print("Saved to notebooks/models/. Copy these into app/backend/myapp/ml_artifacts/ to redeploy.")